# AI-Powered Personal Task Manager

A minimal notebook workflow using Mistral AI, LangChain, LangGraph, MCP, SQLite, and Streamlit.

## 1. Imports and environment

In [6]:
import os
import sys
import json
import sqlite3
import asyncio
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

from langchain_mistralai import ChatMistralAI
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import pandas

load_dotenv()
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
if not MISTRAL_API_KEY:
    raise ValueError("MISTRAL_API_KEY is missing. Add it to your .env file.")

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "tasks.db"
SERVER_PATH = PROJECT_DIR / "task_server.py"

print("Environment loaded.")
print("Project folder:", PROJECT_DIR)

Environment loaded.
Project folder: /Users/sakshamchauhan/Desktop/Task_Manager


## 2. Initialize SQLite database

In [7]:
def get_connection():
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    return connection


def initialise_db():
    with get_connection() as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                ID INTEGER PRIMARY KEY AUTOINCREMENT,
                Title TEXT NOT NULL,
                Category TEXT NOT NULL DEFAULT 'Personal',
                Description TEXT NOT NULL DEFAULT '',
                Priority TEXT NOT NULL DEFAULT 'Medium'
                    CHECK(Priority IN ('Low', 'Medium', 'High')),
                due_date TEXT NOT NULL DEFAULT '',
                Status TEXT NOT NULL DEFAULT 'Pending'
                    CHECK(Status IN ('Pending', 'Completed')),
                Created_at TEXT NOT NULL,
                Updated_at TEXT NOT NULL,
                Completion_at TEXT
            )
        """)
    print("Database ready:", DB_PATH)


def current_timestamp():
    return datetime.now().astimezone().isoformat(timespec="seconds")


initialise_db()

Database ready: /Users/sakshamchauhan/Desktop/Task_Manager/tasks.db


## 3. Create the MCP server

In [8]:
server_code = r'''
import sqlite3
from datetime import datetime
from pathlib import Path
from mcp.server.fastmcp import FastMCP

DB_PATH = Path(__file__).resolve().with_name("tasks.db")
mcp = FastMCP("Task Manager")

def get_connection():
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    return connection

def current_timestamp():
    return datetime.now().astimezone().isoformat(timespec="seconds")

@mcp.tool()
def create_task(title: str, category: str = "Personal",
                description: str = "", priority: str = "Medium",
                due_date: str = "") -> dict:
    """Create a task. due_date must be YYYY-MM-DD or empty."""
    if not isinstance(title, str) or not title.strip():
        return {"success": False, "message": "Title cannot be empty."}
    if category not in ["Personal", "Work", "Study", "Finance", "Health", "Shopping", "Other"]:
        return {"success": False, "message": "Invalid category."}
    if priority not in ["Low", "Medium", "High"]:
        return {"success": False, "message": "Invalid priority."}
    if due_date:
        try:
            parsed = datetime.strptime(due_date, "%Y-%m-%d")
            if parsed.strftime("%Y-%m-%d") != due_date:
                raise ValueError
        except ValueError:
            return {"success": False, "message": "Use a valid YYYY-MM-DD date."}
    now = current_timestamp()
    with get_connection() as conn:
        cursor = conn.execute(
            """INSERT INTO tasks
            (Title, Category, Description, Priority, due_date, Status,
             Created_at, Updated_at, Completion_at)
            VALUES (?, ?, ?, ?, ?, 'Pending', ?, ?, NULL)""",
            (title.strip(), category, description, priority, due_date, now, now)
        )
        task_id = cursor.lastrowid
    return {"success": True, "task_id": task_id, "title": title.strip(),
            "category": category, "priority": priority, "due_date": due_date,
            "status": "Pending", "created_at": now}

@mcp.tool()
def list_tasks(status: str = "", category: str = "") -> dict:
    """List tasks; optionally filter by status Pending/Completed and category."""
    query = "SELECT * FROM tasks WHERE 1=1"
    params = []
    if status:
        if status not in ["Pending", "Completed"]:
            return {"success": False, "message": "Status must be Pending or Completed."}
        query += " AND Status = ?"
        params.append(status)
    if category:
        query += " AND Category = ?"
        params.append(category)
    query += " ORDER BY ID DESC"
    with get_connection() as conn:
        rows = conn.execute(query, params).fetchall()
    return {"success": True, "count": len(rows), "tasks": [dict(row) for row in rows]}

@mcp.tool()
def get_task(task_id: int) -> dict:
    """Get full details for one task by ID."""
    with get_connection() as conn:
        row = conn.execute("SELECT * FROM tasks WHERE ID = ?", (task_id,)).fetchone()
    if row is None:
        return {"success": False, "message": f"Task {task_id} not found."}
    return {"success": True, "task": dict(row)}

@mcp.tool()
def update_task(task_id: int, title: str = "", category: str = "",
                description: str = "", priority: str = "",
                due_date: str = "") -> dict:
    """Update supplied task fields. Empty strings mean leave unchanged."""
    updates = {}
    if title:
        if not title.strip():
            return {"success": False, "message": "Title cannot be empty."}
        updates["Title"] = title.strip()
    if category:
        if category not in ["Personal", "Work", "Study", "Finance", "Health", "Shopping", "Other"]:
            return {"success": False, "message": "Invalid category."}
        updates["Category"] = category
    if description:
        updates["Description"] = description
    if priority:
        if priority not in ["Low", "Medium", "High"]:
            return {"success": False, "message": "Invalid priority."}
        updates["Priority"] = priority
    if due_date:
        try:
            parsed = datetime.strptime(due_date, "%Y-%m-%d")
            if parsed.strftime("%Y-%m-%d") != due_date:
                raise ValueError
        except ValueError:
            return {"success": False, "message": "Use a valid YYYY-MM-DD date."}
        updates["due_date"] = due_date
    if not updates:
        return {"success": False, "message": "No updates provided."}
    now = current_timestamp()
    with get_connection() as conn:
        exists = conn.execute("SELECT ID FROM tasks WHERE ID = ?", (task_id,)).fetchone()
        if exists is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        assignments = ", ".join(f"{field} = ?" for field in updates)
        conn.execute(f"UPDATE tasks SET {assignments}, Updated_at = ? WHERE ID = ?",
                     (*updates.values(), now, task_id))
    return {"success": True, "task_id": task_id, "updated_fields": list(updates),
            "message": "Task updated."}

@mcp.tool()
def complete_task(task_id: int) -> dict:
    """Mark a pending task completed and save completion timestamp."""
    now = current_timestamp()
    with get_connection() as conn:
        row = conn.execute("SELECT Status FROM tasks WHERE ID = ?", (task_id,)).fetchone()
        if row is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        if row["Status"] == "Completed":
            return {"success": False, "message": "Task is already completed."}
        conn.execute("""UPDATE tasks SET Status = 'Completed',
                     Completion_at = ?, Updated_at = ? WHERE ID = ?""",
                     (now, now, task_id))
    return {"success": True, "task_id": task_id, "completion_at": now}

@mcp.tool()
def delete_task(task_id: int) -> dict:
    """Permanently delete a task by ID."""
    with get_connection() as conn:
        row = conn.execute("SELECT Title FROM tasks WHERE ID = ?", (task_id,)).fetchone()
        if row is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        title = row["Title"]
        conn.execute("DELETE FROM tasks WHERE ID = ?", (task_id,))
    return {"success": True, "task_id": task_id, "title": title,
            "message": "Task deleted."}

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
SERVER_PATH.write_text(server_code, encoding="utf-8")
print("MCP server written to:", SERVER_PATH)

MCP server written to: /Users/sakshamchauhan/Desktop/Task_Manager/task_server.py


## 4. Connect Mistral, LangGraph, and MCP

In [ ]:
model = ChatMistralAI(
    model="ministral-3b-2512",
    api_key=MISTRAL_API_KEY,
    temperature=0
)

client = MultiServerMCPClient({
    "task_manager": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(SERVER_PATH)]
    }
})

async def load_agent():
    tools = await client.get_tools()
    agent = create_react_agent(
        model,
        tools,
        prompt=(
            "You are a personal task manager. Use tools for all task operations. "
            "Infer categories and priorities from context, but ask a concise follow-up "
            "when a required detail is ambiguous. Convert relative due dates using "
            "today's date: " + datetime.now().astimezone().date().isoformat() + ". "
            "Pass dates as YYYY-MM-DD. Never claim an operation succeeded unless a tool confirms it. "
            "For task updates or completion, identify the correct task first; ask for clarification "
            "if multiple tasks match. Do not delete tasks unless the user explicitly asks."
        )
    )
    return tools, agent

tools, agent = await load_agent()
print("MCP tools loaded:", [tool.name for tool in tools])

MCP tools loaded: ['create_task', 'list_tasks', 'get_task', 'update_task', 'complete_task', 'delete_task']


/var/folders/cy/sgcgtsjj4ns31pvk0g2rrs5h0000gn/T/ipykernel_45231/2853857817.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 5. Try a natural-language request

In [10]:
async def ask_task_manager(message):
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": message}]
    })
    return result["messages"][-1].content

# Example: change this message to test your agent.
response = await ask_task_manager("Show me all my pending tasks.")
print(response)

Here’s your pending task:

- **Title:** Prepare assignment
  - **Category:** Study
  - **Priority:** High
  - **Due Date:** 2026-10-25


## 6. Create a Streamlit UI

In [14]:
from pathlib import Path

# Set this to your actual project directory if PROJECT_DIR
# has not already been defined.
PROJECT_DIR = Path.cwd()

streamlit_code = r'''
import os
import sys
import asyncio
import logging
from pathlib import Path
from datetime import datetime

import streamlit as st
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient


st.set_page_config(
    page_title="AI Task Manager",
    page_icon="✅",
    layout="centered",
)

logging.basicConfig(level=logging.INFO)

BASE_DIR = Path(__file__).resolve().parent
load_dotenv(BASE_DIR / ".env")

API_KEY = os.getenv("MISTRAL_API_KEY")
SERVER_PATH = BASE_DIR / "task_server.py"
MODEL_NAME = "ministral-3b-2512"


if not API_KEY:
    st.error("MISTRAL_API_KEY is missing from your .env file.")
    st.stop()

if not SERVER_PATH.is_file():
    st.error(f"MCP server not found: {SERVER_PATH}")
    st.stop()


@st.cache_resource
def get_model():
    return ChatMistralAI(
        model=MODEL_NAME,
        api_key=API_KEY,
        temperature=0,
    )


model = get_model()


def get_system_prompt():
    today = datetime.now().astimezone().date().isoformat()

    return f"""
You are an AI-powered personal task manager.
Today's date is {today}.

Use the provided MCP tools for all database operations.
Never invent task records, IDs, or tool results.
Never claim success unless a tool confirms the operation.

CREATION:
Extract the title, description, category, priority, and due date.
Infer category and priority when clear. Ask only essential questions.
Use YYYY-MM-DD dates when calling tools.

RETRIEVAL:
Use tools to retrieve actual tasks and their details.
Filter by category and status when the tools support it.
Use the real category names stored in the database.
Return actual task IDs and stored values.

IDENTIFICATION:
Find existing tasks by ID or title.
If multiple tasks match, ask the user to choose.
Never invent an ID or create a replacement for a missing task.

CONVERSATION:
Use previous messages to understand follow-up requests.
If you ask for a task ID and the user replies with a number,
interpret it as the answer to that question.
Continue the original operation without asking again.
Do not treat a short follow-up as an unrelated new request.

UPDATES:
Identify the correct task before updating it.
Change only the requested fields and preserve all others.
For multiple requested changes, update all requested fields.
For example, adding a description does not require asking for
category, priority, or due date again.
Verify saved changes when possible.

COMPLETION AND DELETION:
Use the appropriate tools for completion and deletion.
Never confuse completing a task with deleting it.
Confirm success only after tool confirmation.
Verify the resulting state when possible.

ERRORS:
If a task cannot be found, say so clearly.
If a tool fails, explain the failure honestly.
Never expose API keys or secrets.

RESPONSE:
Be concise and clear.
For task lists, include IDs, titles, categories, priorities,
due dates, and statuses when available.
Ask clarification only when the intended operation or target
cannot be resolved safely from context and available records.
"""


async def run_agent(messages):
    client = MultiServerMCPClient({
        "task_manager": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(SERVER_PATH)],
        }
    })

    tools = await client.get_tools()

    if not tools:
        raise RuntimeError("The MCP server exposes no tools.")

    agent = create_react_agent(
        model=model,
        tools=tools,
        prompt=get_system_prompt(),
    )

    result = await agent.ainvoke(
        {"messages": messages},
        config={"recursion_limit": 20},
    )

    response_messages = result.get("messages", [])

    if not response_messages:
        raise RuntimeError("The agent returned no messages.")

    content = response_messages[-1].content

    if isinstance(content, str):
        answer = content.strip()
    elif isinstance(content, list):
        answer = "\n".join(
            block["text"]
            for block in content
            if isinstance(block, dict)
            and block.get("type") == "text"
        ).strip()
    else:
        answer = str(content).strip()

    if not answer:
        raise RuntimeError("The agent returned an empty response.")

    return answer


st.title("AI-Powered Personal Task Manager")
st.caption("Manage your tasks using natural language.")

with st.sidebar:
    st.subheader("Task Manager")
    st.caption(f"Model: {MODEL_NAME}")
    st.caption(
        f"Today: {datetime.now().astimezone().date()}"
    )

    if st.button("Clear chat history", use_container_width=True):
        st.session_state.messages = []
        st.rerun()


if "messages" not in st.session_state:
    st.session_state.messages = []


for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])


user_input = st.chat_input(
    "e.g. Add a high-priority Study task due tomorrow"
)

if user_input:
    st.session_state.messages.append({
        "role": "user",
        "content": user_input,
    })

    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Working on your tasks..."):
            try:
                answer = asyncio.run(
                    run_agent(st.session_state.messages)
                )
            except Exception:
                logging.exception("Agent execution failed.")
                answer = (
                    "The request failed. Check the terminal logs "
                    "for the underlying error before retrying."
                )

            st.markdown(answer)

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
    })
'''

app_path = PROJECT_DIR / "app.py"
app_path.write_text(streamlit_code, encoding="utf-8")

# Validate Python syntax before running Streamlit.
compile(streamlit_code, str(app_path), "exec")

print(f"App written and syntax-validated: {app_path}")
print("Next: streamlit run app.py")


App written and syntax-validated: /Users/sakshamchauhan/Desktop/Task_Manager/app.py
Next: streamlit run app.py


## 7. Basic database check

In [12]:
with get_connection() as connection:
    rows = connection.execute(
        "SELECT ID, Title, Category, Priority, due_date, Status, Created_at, Updated_at, Completion_at "
        "FROM tasks ORDER BY ID DESC"
    ).fetchall()

pandas.DataFrame([dict(row) for row in rows]) if rows else print("No tasks yet.")

,ID,Title,Category,Priority,due_date,Status,Created_at,Updated_at,Completion_at
0,2,Prepare assignment,Study,High,2026-10-25,Pending,2026-10-20T18:17:58+05:30,2026-10-20T18:17:58+05:30,NaN
1,1,Revise Java Recursion,Study,Low,2026-10-25,Completed,2026-10-20T18:09:01+05:30,2026-10-20T18:12:14+05:30,2026-10-20T18:12:14+05:30


In [13]:

import sqlite3
import os

db_path = os.path.abspath("tasks.db")

print("Working directory:", os.getcwd())
print("Database path:", db_path)
print("Database exists:", os.path.exists(db_path))

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    # Check the database tables
    tables = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
    """).fetchall()

    print("\nTables:")
    for table in tables:
        print("-", table["name"])

    # Inspect the tasks table
    columns = conn.execute(
        "PRAGMA table_info(tasks)"
    ).fetchall()

    print("\nTask columns:")
    for column in columns:
        print(column["name"], ":", column["type"])

    # Read all tasks directly from SQLite
    if any(t["name"] == "tasks" for t in tables):
        tasks = conn.execute("""
            SELECT *
            FROM tasks
            ORDER BY id
        """).fetchall()

        print("\nActual database records:")
        for task in tasks:
            print(dict(task))

    conn.close()
else:
    print("Database not found at this path.")


Working directory: /Users/sakshamchauhan/Desktop/Task_Manager
Database path: /Users/sakshamchauhan/Desktop/Task_Manager/tasks.db
Database exists: True

Tables:
- tasks
- sqlite_sequence
- task_history

Task columns:
ID : INTEGER
Title : TEXT
Category : TEXT
Description : TEXT
Priority : TEXT
due_date : TEXT
Status : TEXT
Created_at : TEXT
Updated_at : TEXT
Completion_at : TEXT

Actual database records:
{'ID': 2, 'Title': 'Prepare assignment', 'Category': 'Study', 'Description': '', 'Priority': 'High', 'due_date': '2026-10-25', 'Status': 'Pending', 'Created_at': '2026-10-20T18:17:58+05:30', 'Updated_at': '2026-10-20T18:17:58+05:30', 'Completion_at': None}
{'ID': 3, 'Title': 'Finish Project Report', 'Category': 'Work', 'Description': '', 'Priority': 'High', 'due_date': '2026-10-25', 'Status': 'Pending', 'Created_at': '2026-10-20T19:48:32+05:30', 'Updated_at': '2026-10-20T19:48:32+05:30', 'Completion_at': None}
{'ID': 4, 'Title': 'Python Assignment', 'Category': 'Study', 'Description': 'C